In [70]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import Filter, FieldCondition, MatchValue
from qdrant_client.models import Distance, VectorParams, PointStruct
from tqdm import tqdm

from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csgraph
from sklearn.manifold import SpectralEmbedding
from sklearn.preprocessing import normalize

import numpy as np
import torch
from PIL import Image
import os
import pandas as pd

#importing the used models
from transformers import BlipProcessor, BlipModel

from utils.metrics import iterate_ds, mean_reciprocal_rank, recall_at_k

In [42]:
def load_and_filter_data(caption_csv_path):
    img_dir="./images"
    img_prefix="img_"
    df = pd.read_csv(caption_csv_path, encoding='latin1')
    df['image_filename'] = [f"{img_prefix}{i}.jpg" for i in df.index]
    df['full_image_path'] = [f"{img_dir}/{fname}" for fname in df['image_filename']]
    df_valid = df[df['full_image_path'].apply(os.path.exists)].reset_index(drop=True)
    print(f"Valid images with captions: {len(df_valid)}")
    return df_valid


def generate_image_embeddings(image_paths, processor, model, device):
    embeddings = []
    for path in image_paths:
        try:
            image = Image.open(path).convert("RGB")
            inputs = processor(images=image, return_tensors="pt").to(device)
            with torch.no_grad():
                features = model.get_image_features(**inputs)
            features = features / features.norm(p=2, dim=-1, keepdim=True)
            embeddings.append(features.cpu().numpy().squeeze())
        except Exception as e:
            print(f"Error processing image {path}: {e}")
    return np.array(embeddings)

def generate_text_embeddings(texts, processor, model, device):
    embeddings = []
    for text in texts:
        try:
            inputs = processor(text=text, return_tensors="pt").to(device)
            with torch.no_grad():
                features = model.get_text_features(**inputs)
            features = features / features.norm(p=2, dim=-1, keepdim=True)
            embeddings.append(features.cpu().numpy().squeeze())
        except Exception as e:
            print(f"Error processing text '{text[:30]}...': {e}")
    return np.array(embeddings)

def match_embeddings(image_embeddings, text_embeddings, valid_image_paths, valid_captions):
    similarity_matrix = cosine_similarity(image_embeddings, text_embeddings)
    print(similarity_matrix)
    matched_pairs = []
    for i in range(similarity_matrix.shape[0]):
        j = np.argmax(similarity_matrix[i])
        matched_pairs.append({
            "image_path": valid_image_paths[i],
            "caption": valid_captions[j],
            "image_embedding": image_embeddings[i],
            "text_embedding": text_embeddings[j],
            "similarity": similarity_matrix[i, j]
        })
    print(f"Matched {len(matched_pairs)} image-caption pairs via cosine similarity.")
    return matched_pairs, similarity_matrix

In [75]:
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipModel.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

df_long = load_and_filter_data("./long_generated_captions.csv")
df_short = load_and_filter_data("./short_generated_captions.csv")

valid_image_paths = df_long['full_image_path'].tolist()
valid_captions_long = df_long['paraphrased_caption'].tolist()
valid_captions_short = df_short['paraphrased_caption'].tolist()

image_embeddings = generate_image_embeddings(valid_image_paths, processor, model, device)
text_embeddings_long = generate_text_embeddings(valid_captions_long, processor, model, device)
text_embeddings_short = generate_text_embeddings(valid_captions_short, processor, model, device)
matched_pairs_long, similarity_matrix_spectral_long = match_embeddings(image_embeddings, text_embeddings_long, valid_image_paths, valid_captions_long)
matched_pairs_short, similarity_matrix_spectral_short = match_embeddings(image_embeddings, text_embeddings_short, valid_image_paths, valid_captions_short)


`BlipModel` is going to be deprecated in future release, please use `BlipForConditionalGeneration`, `BlipForQuestionAnswering` or `BlipForImageTextRetrieval` depending on your usecase.
Some weights of BlipModel were not initialized from the model checkpoint at Salesforce/blip-image-captioning-base and are newly initialized: ['logit_scale', 'text_model.embeddings.LayerNorm.bias', 'text_model.embeddings.LayerNorm.weight', 'text_model.embeddings.position_embeddings.weight', 'text_model.embeddings.word_embeddings.weight', 'text_model.encoder.layer.0.attention.output.LayerNorm.bias', 'text_model.encoder.layer.0.attention.output.LayerNorm.weight', 'text_model.encoder.layer.0.attention.output.dense.bias', 'text_model.encoder.layer.0.attention.output.dense.weight', 'text_model.encoder.layer.0.attention.self.key.bias', 'text_model.encoder.layer.0.attention.self.key.weight', 'text_model.encoder.layer.0.attention.self.query.bias', 'text_model.encoder.layer.0.attention.self.query.weight', 'text_mo

Valid images with captions: 498
Valid images with captions: 498


In [52]:
def apply_spectral_embedding_from_cosine(B, n_components=128):
    B = np.clip(B, 0, None)

    n_texts, n_images = B.shape

    zero_text = np.zeros((n_texts, n_texts))
    zero_image = np.zeros((n_images, n_images))
    
    A = np.block([
        [zero_text, B],
        [B.T, zero_image]
    ])

    epsilon = 1e-3
    A += epsilon
    np.fill_diagonal(A, epsilon)

    n_total = A.shape[0]
    if n_components >= n_total:
        n_components = n_total - 1
        print(f"Warning: n_components reduced to {n_components} because it must be less than total nodes {n_total}")

    spectral = SpectralEmbedding(n_components=n_components, affinity='precomputed')
    embedded = spectral.fit_transform(A)

    spectral_text = normalize(embedded[:n_texts])
    spectral_image = normalize(embedded[n_texts:])
    
    return spectral_image, spectral_text


In [24]:
spectral_image_embeddings, spectral_text_long_embeddings = apply_spectral_embedding_from_cosine(similarity_matrix_spectral_long)
spectral_image_embeddings, spectral_text_short_embeddings = apply_spectral_embedding_from_cosine(similarity_matrix_spectral_short)

## Generate and add vector embeddings to Qdrant
**Normal embeddings of original_captions column**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipModel.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

df_valid = load_and_filter_data("./pixel_prose.csv")

`BlipModel` is going to be deprecated in future release, please use `BlipForConditionalGeneration`, `BlipForQuestionAnswering` or `BlipForImageTextRetrieval` depending on your usecase.
Some weights of BlipModel were not initialized from the model checkpoint at Salesforce/blip-image-captioning-base and are newly initialized: ['logit_scale', 'text_model.embeddings.LayerNorm.bias', 'text_model.embeddings.LayerNorm.weight', 'text_model.embeddings.position_embeddings.weight', 'text_model.embeddings.word_embeddings.weight', 'text_model.encoder.layer.0.attention.output.LayerNorm.bias', 'text_model.encoder.layer.0.attention.output.LayerNorm.weight', 'text_model.encoder.layer.0.attention.output.dense.bias', 'text_model.encoder.layer.0.attention.output.dense.weight', 'text_model.encoder.layer.0.attention.self.key.bias', 'text_model.encoder.layer.0.attention.self.key.weight', 'text_model.encoder.layer.0.attention.self.query.bias', 'text_model.encoder.layer.0.attention.self.query.weight', 'text_mo

Valid images with captions: 421


In [36]:
df_valid = df_valid.rename(columns={'Unnamed: 0': "img_idx"})

### Add to Qdrant

In [6]:
df_s_captions = pd.read_csv("./short_generated_captions.csv", encoding='latin1')

In [37]:
valid_captions = df_s_captions['original_caption'].tolist()
img_ids = df_s_captions["id"].tolist()
# needed as our dataset saved from 1 and not 0
valid_img_idx = [id+1 for id in img_ids]
valid_img_paths = [f'./images/img_{id}.jpg' for id in valid_img_idx]

image_embeddings = generate_image_embeddings(valid_img_paths, processor, model, device)
text_embeddings = generate_text_embeddings(valid_captions, processor, model, device)

Error processing image ./images/img_12.jpg: [Errno 2] No such file or directory: './images/img_12.jpg'
Error processing image ./images/img_55.jpg: [Errno 2] No such file or directory: './images/img_55.jpg'
Error processing image ./images/img_56.jpg: [Errno 2] No such file or directory: './images/img_56.jpg'
Error processing image ./images/img_61.jpg: [Errno 2] No such file or directory: './images/img_61.jpg'
Error processing image ./images/img_70.jpg: [Errno 2] No such file or directory: './images/img_70.jpg'
Error processing image ./images/img_74.jpg: [Errno 2] No such file or directory: './images/img_74.jpg'
Error processing image ./images/img_86.jpg: [Errno 2] No such file or directory: './images/img_86.jpg'
Error processing image ./images/img_101.jpg: [Errno 2] No such file or directory: './images/img_101.jpg'
Error processing image ./images/img_112.jpg: [Errno 2] No such file or directory: './images/img_112.jpg'
Error processing image ./images/img_113.jpg: [Errno 2] No such file o

In [47]:
collection_name = "normal_original_captions"

client = QdrantClient(host="localhost", port=6333)

points = [
    PointStruct(id=idx, vector=vec, payload={"type": "image"})
    for idx, vec in zip(valid_img_idx, image_embeddings)
]

client.upsert(collection_name=collection_name, points=points)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [ ]:
client = QdrantClient(url="http://localhost:6333")

df_short = pd.read_csv("./short_generated_captions.csv", encoding='latin1')
df_long = pd.read_csv("./long_generated_captions.csv", encoding='latin1')

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipModel.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

datasets = [("Short Captions", df_short), ("Long Captions", df_long)]
metrics = ["mrr", "recall@k"]

for name, df in datasets:
    for metric in metrics:
        avg_score = iterate_ds(3, df, metric, processor, model, "paraphrased_caption", client, "normal_original_captions", device)
        print(f"Average {metric.upper()} for {name}: {avg_score:.4f}")

`BlipModel` is going to be deprecated in future release, please use `BlipForConditionalGeneration`, `BlipForQuestionAnswering` or `BlipForImageTextRetrieval` depending on your usecase.
Some weights of BlipModel were not initialized from the model checkpoint at Salesforce/blip-image-captioning-base and are newly initialized: ['logit_scale', 'text_model.embeddings.LayerNorm.bias', 'text_model.embeddings.LayerNorm.weight', 'text_model.embeddings.position_embeddings.weight', 'text_model.embeddings.word_embeddings.weight', 'text_model.encoder.layer.0.attention.output.LayerNorm.bias', 'text_model.encoder.layer.0.attention.output.LayerNorm.weight', 'text_model.encoder.layer.0.attention.output.dense.bias', 'text_model.encoder.layer.0.attention.output.dense.weight', 'text_model.encoder.layer.0.attention.self.key.bias', 'text_model.encoder.layer.0.attention.self.key.weight', 'text_model.encoder.layer.0.attention.self.query.bias', 'text_model.encoder.layer.0.attention.self.query.weight', 'text_mo

Average MRR for Short Captions: 0.0028
Average RECALL@K for Short Captions: 0.0050
Average MRR for Long Captions: 0.0033
Average RECALL@K for Long Captions: 0.0050


In [71]:
def get_image_embeddings_from_qdrant(client: QdrantClient, collection_name: str, ids: list[int], dim=512):
    image_embeddings = []
    for idx in tqdm(ids, desc="Retrieving image embeddings"):
        result = client.retrieve(
            collection_name=collection_name,
            ids=[idx],
            with_payload=False
        )
        if result and result[0].vector is not None:
            image_embeddings.append(result[0].vector)
        else:
            image_embeddings.append(np.zeros(dim))
    return np.array(image_embeddings)


In [72]:
def get_text_embeddings(df: pd.DataFrame, processor, model, device: str, caption_column: str):
    text_embeddings = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Computing text embeddings"):
        caption = row[caption_column]
        inputs = processor(text=caption, return_tensors="pt").to(device)
        with torch.no_grad():
            features = model.get_text_features(**inputs)
        features = features / features.norm(p=2, dim=-1, keepdim=True)
        text_embeddings.append(features.cpu().numpy().squeeze())
    return np.array(text_embeddings)

In [74]:
def mean_reciprocal_rank_spectral(k, spectral_text, spectral_image):
    sim = cosine_similarity(spectral_text, spectral_image)
    total = 0
    for idx in range(sim.shape[0]):
        top_k = np.argsort(sim[idx])[::-1]
        if idx in top_k[:k]:
            rank = np.where(top_k == idx)[0][0] + 1
            total += 1 / rank
    return total / sim.shape[0]

In [75]:
def recall_at_k_spectral(k, spectral_text, spectral_image):
    sim = cosine_similarity(spectral_text, spectral_image)
    total = 0
    for idx in range(sim.shape[0]):
        top_k = np.argsort(sim[idx])[-k:][::-1]
        if idx in top_k:
            total += 1
    return total / sim.shape[0]

In [78]:
client = QdrantClient(host="localhost", port=6333)

for name, df in [("Short", df_short), ("Long", df_long)]:
        print(f"\nProcessing {name} Captions")
        ids = df.index.tolist()

        image_embeddings = get_image_embeddings_from_qdrant(client, collection_name, ids, dim=512)
        text_embeddings = get_text_embeddings(df, processor, model, device, "paraphrased_caption")

        B = cosine_similarity(text_embeddings, image_embeddings)

        spectral_text, spectral_image = apply_spectral_embedding_from_cosine(B, n_components=128)

        k = 3
        mrr = mean_reciprocal_rank_spectral(k, spectral_text, spectral_image)
        recall = recall_at_k_spectral(k, spectral_text, spectral_image)

        print(f"  Spectral MRR:      {mrr:.4f}")
        print(f"  Spectral Recall@{k}: {recall:.4f}")




Processing Short Captions


Computing text embeddings: 100%|██████████| 600/600 [01:11<00:00,  8.42it/s]


  Spectral MRR:      0.0078
  Spectral Recall@3: 0.0117

Processing Long Captions


Computing text embeddings: 100%|██████████| 600/600 [02:09<00:00,  4.64it/s]


  Spectral MRR:      0.0008
  Spectral Recall@3: 0.0017
